# `kitai.index` — User Guide

`kitai/index.py` provides three groups of functionality:

```
kitai/index.py
│
├── Embedding utilities          (no network required)
│   ├── get_embedding_dim()      ndarray → int
│   └── load_embeddings_from_csv()  CSV → ndarray
│
├── Retrieval
│   ├── create_vectorstore()     docs + embeddings → FAISS
│   └── create_BM25retriever_from_docs()  docs → BM25Retriever
│
└── Disk-based batch pipeline    (OpenAI Batch API, folder-based)
    ├── create_batch_files_embeddings()   docs → JSONL files on disk
    ├── create_embeddings_batches()       folder → batch job IDs
    └── retrieve_embeddings_batches()     job IDs → (custom_id, embedding) pairs
```

**Prerequisites:**
- Sections 1–4 run with **no API key** — all examples use synthetic data.
- Section 5 (batch pipeline) requires an `OPENAI_API_KEY` in your `.env`.

**Related modules:**
- `kitai.batch` — modern, in-memory batch API helpers (no JSONL folder required).
- `utils` — document loading helpers (`load_text_file`, `chunk_documents`, …).

## Sections
1. [Setup](#1-setup)
2. [Embedding utilities](#2-embedding-utilities)
3. [FAISS vector store](#3-faiss-vector-store)
4. [BM25 retriever](#4-bm25-retriever)
5. [Disk-based batch pipeline](#5-disk-based-batch-pipeline)
6. [Error handling reference](#6-error-handling-reference)

---
## 1 — Setup

In [ ]:
import logging
import numpy as np
from langchain_core.documents import Document

from kitai.index import (
    get_embedding_dim,
    load_embeddings_from_csv,
    create_vectorstore,
    create_BM25retriever_from_docs,
    create_batch_files_embeddings,
    create_embeddings_batches,
    retrieve_embeddings_batches,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

### Sample documents used throughout this guide

All examples share this small corpus.  Every document must carry
`metadata["id"]` — a unique value used as the key inside the vector store
and in the batch pipeline's `custom_id` field.

In production, use `utils.load_text_file` / `chunk_documents` /
`assign_sequential_ids` to build this list from a real file.

In [ ]:
DOCS = [
    Document(page_content="Short selling profits when a stock price falls.",   metadata={"id": 1}),
    Document(page_content="Alpha measures excess return over a benchmark.",     metadata={"id": 2}),
    Document(page_content="Drawdown is the peak-to-trough decline in a portfolio.", metadata={"id": 3}),
    Document(page_content="Risk management is the foundation of any strategy.", metadata={"id": 4}),
    Document(page_content="Volatility measures the dispersion of returns.",     metadata={"id": 5}),
]

DIM = 8  # small synthetic dimension for offline demos

# Deterministic synthetic embeddings — one row per document
rng = np.random.default_rng(seed=42)
EMBEDDINGS = rng.random((len(DOCS), DIM)).astype(np.float32)

print(f"Docs     : {len(DOCS)}")
print(f"Embedding shape: {EMBEDDINGS.shape}  (n_docs × dim)")

---
## 2 — Embedding utilities

Two helpers for inspecting and loading pre-computed embeddings.
Neither makes a network call.

### `get_embedding_dim`

Extracts the dimensionality from a 2-D embedding array.
Validates that the input is a numpy ndarray with exactly two dimensions.

In [ ]:
dim = get_embedding_dim(EMBEDDINGS)
print(f"Embedding dimension: {dim}")

# Works on real-scale arrays too
large = np.zeros((1000, 1536), dtype=np.float32)
print(f"1536-dim array     : {get_embedding_dim(large)}")

### `load_embeddings_from_csv`

Reads a CSV file where one column contains embeddings stored as
Python list strings (e.g. `"[0.1, 0.2, 0.3, ...]"`), parses each row,
and returns a stacked `np.ndarray` of shape `(n_rows, embedding_dim)`.

This is the format produced by `kitai.batch.parse_embedding_results`
when you save with `pd.DataFrame.to_csv`.

In [ ]:
import pandas as pd, tempfile, os

# Create a temporary CSV that mimics the book_embeddings.csv format
sample_df = pd.DataFrame({
    "custom_id": [f"custom_id_{i+1}" for i in range(len(DOCS))],
    "embedding": [str(row.tolist()) for row in EMBEDDINGS],
})
tmp_csv = tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w")
sample_df.to_csv(tmp_csv.name, index=False)
tmp_csv.close()

print("Sample CSV preview:")
print(sample_df.to_string(index=False))

In [ ]:
loaded_embeddings = load_embeddings_from_csv(
    path_to_csv=tmp_csv.name,
    embedding_column="embedding",
)

print(f"Shape    : {loaded_embeddings.shape}")
print(f"dtype    : {loaded_embeddings.dtype}")
print(f"First row: {loaded_embeddings[0]}")

os.unlink(tmp_csv.name)  # clean up

# To load your real embeddings file:
# embeddings = load_embeddings_from_csv("./book_embeddings.csv")

---
## 3 — FAISS vector store

### How `create_vectorstore` works

```
docs + embeddings
       │
       ├─ faiss.IndexFlatL2(dim)     ← raw L2 index over embedding vectors
       ├─ InMemoryDocstore           ← documents keyed by metadata["id"]
       └─ index_to_docstore_id map   ← FAISS row index → metadata["id"]
              │
              └─► FAISS vector store  (LangChain wrapper)
```

**Two invariants that must hold:**
1. `len(docs) == embeddings.shape[0]` — i-th doc must correspond to i-th embedding row.
2. Every `doc.metadata["id"]` must be unique — it is the docstore key.

**The `fake_embeddings_model` parameter:**  
FAISS stores an embedding function for re-encoding *query strings* at
search time.  Since the document embeddings are already computed, this
model is **not** called during construction — only when you call
`similarity_search("some query text")`.  For demos without an API key,
use `FakeEmbeddings`; in production use `OpenAIEmbeddings`.

In [ ]:
from langchain_core.embeddings import FakeEmbeddings

# FakeEmbeddings returns random vectors of the specified size.
# Replace with OpenAIEmbeddings() when you need real semantic search.
query_encoder = FakeEmbeddings(size=DIM)

vector_store = create_vectorstore(
    docs=DOCS,
    embeddings=EMBEDDINGS,
    fake_embeddings_model=query_encoder,
)

print(f"Vector store contains {vector_store.index.ntotal} vectors")
print(f"Docstore keys        : {list(vector_store.docstore._dict.keys())}")

### Searching the vector store

FAISS exposes two search methods:

| Method | Input | When to use |
|---|---|---|
| `similarity_search_by_vector(vec, k)` | `list[float]` | You already have the query embedding |
| `similarity_search(query, k)` | `str` | FAISS re-encodes the string using `fake_embeddings_model` |

In [ ]:
# Search by vector — no embedding model called
query_vec = EMBEDDINGS[0].tolist()   # use doc 1's own embedding as the query

results = vector_store.similarity_search_by_vector(query_vec, k=3)

print("Top-3 results (search by vector):")
for i, doc in enumerate(results, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

In [ ]:
# Search by string — calls fake_embeddings_model to encode the query
# With FakeEmbeddings the ranking is random; replace with OpenAIEmbeddings
# for semantic results.
results_str = vector_store.similarity_search("what is drawdown?", k=3)

print("Top-3 results (search by string):")
for i, doc in enumerate(results_str, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

### Retrieving scores alongside results

In [ ]:
results_with_scores = vector_store.similarity_search_with_score(
    "what is drawdown?", k=3
)

print("Results with L2 distance scores (lower = more similar):")
for doc, score in results_with_scores:
    print(f"  score={score:.4f}  [id={doc.metadata['id']}] {doc.page_content}")

---
## 4 — BM25 retriever

BM25 is a keyword-based (sparse) retrieval algorithm.  It ranks documents
by term frequency and inverse document frequency — no embeddings needed.

**When to choose BM25 over FAISS:**
- Exact keyword matches matter more than semantic similarity.
- You don't have pre-computed embeddings yet.
- You want a fast, cheap baseline for hybrid retrieval.

**Hybrid retrieval pattern** (combine both):
```python
from kitai.retriever import create_hybrid_retriever

hybrid = create_hybrid_retriever(
    sparse_retriever=bm25_retriever,
    semantic_retriever=faiss_retriever,
    weights_sparse=0.5,   # BM25 gets 0.5, FAISS gets 0.5
)
```
See `retriever_guide.ipynb` for a full walkthrough of `create_hybrid_retriever`.

In [ ]:
bm25_retriever = create_BM25retriever_from_docs(docs=DOCS, k=3)

print(f"BM25 retriever ready. k={bm25_retriever.k}")
print(f"Corpus size: {len(bm25_retriever.docs)} documents")

In [ ]:
query = "risk and volatility"
bm25_results = bm25_retriever.invoke(query)

print(f"Query    : {query!r}")
print(f"Results  : {len(bm25_results)}")
for i, doc in enumerate(bm25_results, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

---
## 5 — Disk-based batch pipeline

This pipeline writes task files to disk before uploading — useful when you
want to inspect or version-control the batch input before sending it to OpenAI.

```
create_batch_files_embeddings()  ─►  ./batch_files/*.jsonl  (on disk)
         │
create_embeddings_batches()      ─►  job_ids  (uploaded + submitted)
         │
    [wait for completion]             check status with client.batches.retrieve()
         │
retrieve_embeddings_batches()    ─►  [(custom_id, embedding), ...]
         │
create_vectorstore()             ─►  FAISS vector store
```

> **Note:** For a simpler in-memory workflow that skips the JSONL folder,
> use `kitai.batch.build_embedding_tasks` + `submit_batch_job` instead.
> See `batch_api_guide.ipynb`.

### Step 1 — Write JSONL batch files to disk

`create_batch_files_embeddings` splits the document list into files of
`batch_size` rows each and writes them as JSONL under `output_dir`.
Existing files with the same name are overwritten.

- `batch_size=20_000` is the OpenAI per-file limit for the Batch API.
- Adjust `batch_file_name` to namespace files for different datasets.

In [ ]:
import tempfile, os, json

batch_dir = tempfile.mkdtemp(prefix="kitai_batch_")

create_batch_files_embeddings(
    docs=DOCS,
    batch_size=3,              # small so we get 2 files for this demo
    batch_file_name="demo_batch",
    output_dir=batch_dir,
)

# Inspect what was written
files = sorted(os.listdir(batch_dir))
print(f"Files created: {files}")

print("\nFirst file — first 2 lines:")
with open(os.path.join(batch_dir, files[0])) as f:
    for line in list(f)[:2]:
        task = json.loads(line)
        print(f"  custom_id={task['custom_id']}  input={task['body']['input'][:50]}...")

### Step 2 — Upload files and submit batch jobs

`create_embeddings_batches` reads every file in `batch_folder`, uploads
each to the OpenAI Files API, and creates one batch job per file.
It returns the list of batch job IDs.

Failures during upload or job creation are logged at ERROR and skipped
(partial-failure tolerant).

> **Requires OPENAI_API_KEY.** The cell below is guarded so it won't run
> if the key is missing.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping submission.")
else:
    client = OpenAI()

    job_ids = create_embeddings_batches(
        client=client,
        batch_folder=batch_dir,
        completion_window="24h",
    )

    print(f"Submitted {len(job_ids)} job(s):")
    for jid in job_ids:
        print(f"  {jid}")

    # Persist for the next cell
    # job_ids = ["batch_abc", "batch_def"]

### Step 3 — Wait for jobs to complete

Use `client.batches.retrieve(job_id)` to poll status.
For automated polling, prefer `kitai.batch.poll_until_complete`.

Typical status progression:
```
validating → in_progress → finalizing → completed
```

In [ ]:
# Manual single-shot status check
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping status check.")
else:
    for job_id in job_ids:
        batch = client.batches.retrieve(job_id)
        counts = batch.request_counts
        print(
            f"{job_id}  status={batch.status}  "
            f"completed={counts.completed}/{counts.total}"
        )

    # For blocking poll, use kitai.batch:
    # from kitai.batch import poll_until_complete
    # completed_ids = poll_until_complete(client, job_ids, poll_interval=10.0)

### Step 4 — Retrieve embeddings from completed jobs

`retrieve_embeddings_batches` takes job IDs, downloads the output JSONL
for each, and returns a flat list of `(custom_id, embedding)` tuples.

Failures at the job, file, or line level are logged and skipped.  A
completion summary is logged at INFO:
```
Retrieved N embeddings. Failed jobs: X, failed files: Y
```

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping retrieval.")
else:
    embedding_pairs = retrieve_embeddings_batches(client, job_ids)

    print(f"Retrieved {len(embedding_pairs)} (custom_id, embedding) pairs")
    cid, vec = embedding_pairs[0]
    print(f"Example  : custom_id={cid!r}  dim={len(vec)}  first_values={vec[:3]}")

### Step 5 — Assemble the FAISS vector store

Once you have `embedding_pairs`, convert them to a numpy array in the
same order as `DOCS` and pass both to `create_vectorstore`.

In [ ]:
from langchain_core.embeddings import FakeEmbeddings as _Fake

if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — showing pattern with synthetic data.")
    embedding_pairs = [(f"custom_id_{doc.metadata['id']}", row.tolist())
                       for doc, row in zip(DOCS, EMBEDDINGS)]

# Sort pairs to match DOCS order using the numeric ID in custom_id
id_to_vec = {
    int(cid.split("custom_id_")[1]): vec
    for cid, vec in embedding_pairs
}
ordered_vecs = np.array(
    [id_to_vec[doc.metadata["id"]] for doc in DOCS], dtype=np.float32
)

query_encoder = _Fake(size=ordered_vecs.shape[1])
vs = create_vectorstore(DOCS, ordered_vecs, query_encoder)

print(f"Vector store ready. Vectors indexed: {vs.index.ntotal}")

# Clean up the temp batch dir
import shutil
shutil.rmtree(batch_dir, ignore_errors=True)

---
## 6 — Error handling reference

Every function guards its inputs and raises before any expensive work.

In [ ]:
# get_embedding_dim — wrong type and wrong shape
for bad, label in [
    ([[1, 2], [3, 4]],            "plain list (not ndarray)"),
    (np.zeros(16),                "1-D array"),
    (np.zeros((2, 4, 8)),         "3-D array"),
]:
    try:
        get_embedding_dim(bad)
    except (TypeError, ValueError) as e:
        print(f"{label}:\n  {type(e).__name__}: {e}\n")

In [ ]:
# create_vectorstore — mismatched lengths
wrong_embeddings = np.zeros((len(DOCS) + 2, DIM), dtype=np.float32)
try:
    create_vectorstore(DOCS, wrong_embeddings, None)
except ValueError as e:
    print(f"Mismatched lengths:\n  {e}\n")

In [ ]:
# create_BM25retriever_from_docs — empty list and non-positive k
for args, label in [
    (([], 5),    "empty docs"),
    ((DOCS, 0),  "k = 0"),
    ((DOCS, -1), "k = -1"),
]:
    try:
        create_BM25retriever_from_docs(*args)
    except ValueError as e:
        print(f"{label}:\n  {e}\n")

In [ ]:
# create_batch_files_embeddings — empty docs and zero batch_size
import tempfile
tmpdir = tempfile.mkdtemp()

for kwargs, label in [
    (dict(docs=[], batch_size=100, output_dir=tmpdir),         "empty docs"),
    (dict(docs=DOCS, batch_size=0, output_dir=tmpdir),         "batch_size = 0"),
    (dict(docs=DOCS, batch_size=-1, output_dir=tmpdir),        "batch_size = -1"),
]:
    try:
        create_batch_files_embeddings(**kwargs)
    except ValueError as e:
        print(f"{label}:\n  {e}\n")

shutil.rmtree(tmpdir)

In [ ]:
# load_embeddings_from_csv — missing file and missing column
try:
    load_embeddings_from_csv(path_to_csv="/does/not/exist.csv")
except FileNotFoundError as e:
    print(f"Missing file:\n  {e}\n")

import pandas as pd, tempfile
df_bad = pd.DataFrame({"text": ["hello", "world"]})
bad_csv = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
df_bad.to_csv(bad_csv.name, index=False)
bad_csv.close()

try:
    load_embeddings_from_csv(path_to_csv=bad_csv.name, embedding_column="embedding")
except KeyError as e:
    print(f"Missing column:\n  {e}\n")

os.unlink(bad_csv.name)